In [22]:
import pymongo
from pymongo import MongoClient

def diagnose_mongodb(db_host, db_name, collection_pattern="fbirn"):
    """Diagnose MongoDB collections and document structure"""
    print(f"\n{'='*40}")
    print("MongoDB Database Diagnosis")
    print(f"{'='*40}")
    
    try:
        # 1. Initialize MongoDB client
        client = MongoClient(f"mongodb://{db_host}:27017",
                           serverSelectionTimeoutMS=30000,
                           socketTimeoutMS=30000)
        db = client[db_name]
        
        # 2. List all collections matching pattern
        all_collections = db.list_collection_names()
        target_collections = [col for col in all_collections if collection_pattern in col]
        
        print(f"\nFound {len(target_collections)} collections matching '{collection_pattern}':")
        for col in target_collections:
            count = db[col].count_documents({})
            print(f"- {col}: {count} documents")
            
            # Show first document's keys if collection exists
            if count > 0:
                doc = db[col].find_one()
                print(f"  Sample keys: {list(doc.keys())}")
                if 'id' in doc:
                    print(f"  First document ID: {doc['id']}")
                if 'kind' in doc:
                    print(f"  Document kind: {doc['kind']}")
                
        # 3. Check for .bin variants
        bin_collections = [col for col in all_collections if f"{collection_pattern}" in col and col.endswith('.bin')]
        if bin_collections:
            print("\nFound .bin collections:")
            for col in bin_collections:
                count = db[col].count_documents({})
                print(f"- {col}: {count} documents")
                if count > 0:
                    doc = db[col].find_one()
                    print(f"  Sample keys: {list(doc.keys())}")
        else:
            print("\nNo .bin collections found")

        # 4. Check for gridfs collections
        gridfs_collections = [col for col in all_collections if col.startswith('fs.')]
        if gridfs_collections:
            print("\nFound GridFS collections:")
            for col in gridfs_collections:
                print(f"- {col}")

    except Exception as e:
        print(f"\n✗ Diagnosis failed: {str(e)}")
        raise
    finally:
        client.close()
    print(f"{'='*40}\n")

if __name__ == "__main__":
    diagnose_mongodb(
        db_host="10.245.12.58",
        db_name="multimodalSubnetworks",
        collection_pattern="fbirn"
    )


MongoDB Database Diagnosis

Found 6 collections matching 'fbirn':
- fbirn_falff.meta: 357 documents
  Sample keys: ['_id', 'id', 'collection', 'filename', 'id_formats', 'gender', 'data_types', 'processing_log']
  First document ID: 0
- fbirn_smri.bin: 714 documents
  Sample keys: ['_id', 'id', 'chunk_id', 'kind', 'chunk']
  First document ID: 4
  Document kind: image
- fbirn_smri.meta: 357 documents
  Sample keys: ['_id', 'id', 'collection', 'filename', 'id_formats', 'gender', 'data_types', 'processing_log']
  First document ID: 0
- fbirn_falff.bin: 714 documents
  Sample keys: ['_id', 'id', 'chunk_id', 'kind', 'chunk']
  First document ID: 0
  Document kind: image
- fbirn_dwi.bin: 650 documents
  Sample keys: ['_id', 'id', 'chunk_id', 'kind', 'chunk']
  First document ID: 31
  Document kind: image
- fbirn_dwi.meta: 325 documents
  Sample keys: ['_id', 'id', 'collection', 'filename', 'id_formats', 'gender', 'data_types', 'processing_log']
  First document ID: 31

Found .bin collection